In [122]:
import pandas as pd
import numpy as np
n_items = 500
n_users = 500
k = 10 
ten = [str(j).zfill(2) for j in list(range(1,k+1))]
ten

['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']

In [123]:
labs = ['test'+i for i in ten]
labs.insert(0,'orig')
url = "https://raw.githubusercontent.com/park61/imputation/main/data/"
url_dict = {}
url_dict["orig"] = url+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
for i in ten:
    url_dict["test{0}".format(i)] = url+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'

In [124]:
print(labs)
for lab in labs:
  print(url_dict[lab])

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
https://raw.githubusercontent.com/park61/imputation/main/data/500-by-500_original_matrix.csv
https://raw.githubusercontent.com/park61/imputation/main/data/500-by-500_10_fold_test_data01.csv
https://raw.githubusercontent.com/park61/imputation/main/data/500-by-500_10_fold_test_data02.csv
https://raw.githubusercontent.com/park61/imputation/main/data/500-by-500_10_fold_test_data03.csv
https://raw.githubusercontent.com/park61/imputation/main/data/500-by-500_10_fold_test_data04.csv
https://raw.githubusercontent.com/park61/imputation/main/data/500-by-500_10_fold_test_data05.csv
https://raw.githubusercontent.com/park61/imputation/main/data/500-by-500_10_fold_test_data06.csv
https://raw.githubusercontent.com/park61/imputation/main/data/500-by-500_10_fold_test_data07.csv
https://raw.githubusercontent.com/park61/imputation/main/data/500-by-500_10_fold_test_data08.csv
https://raw.githubuser

In [125]:
df_dict = {}
for lab in labs:
  df = pd.read_csv(url_dict[lab])
  df_dict[lab] = df.set_index('item_id')

In [126]:
# compute sparcity of the original data
#df_dict["orig"].head()
#pivot_df = df_dict["orig"].pivot_table(index="item_id", columns="user_id",values="rating", aggfunc='mean')
n_row = df_dict["orig"].shape[0]
n_col = df_dict["orig"].shape[1]
Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
sparsity = 1 - num_Obs / (n_row * n_col)
print(sparsity)

0.740172


In [127]:
import numpy as np
col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
#print(col_max)

In [128]:
def normal3(df):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  maxs = [int(df[i].max()) for i in range(n)]
  probs = {}
  for j in range(n):
    mm = m - df.isna().sum(axis=0)[j]
    probs[j] = [np.count_nonzero(df[j] == k)/mm for k in range(1,maxs[j]+1)]
  df_out = df.copy()
  for j in range(n):
    if maxs[j] == 2:
      df_out[j] = df[j]-1
    else:
      c = maxs[j]
      for i in range(m):
        r = df.iloc[i,j]
        if np.isnan(r):
          df_out.iloc[i,j] = np.nan
        elif r == 1:
          df_out.iloc[i,j] = 0
        elif r == 2:
          df_out.iloc[i,j] = (probs[j][0]+probs[j][1])/(sum(probs[j][:c-1])+sum(probs[j][1:c]))
        else:
          df_out.iloc[i,j] = (sum(probs[j][:int(r)-1])+sum(probs[j][1:int(r)]))/(sum(probs[j][:int(c)-1])+sum(probs[j][1:int(c)]))
  return df_out#, maxs, probs

In [129]:
def unnormal3(df,maxs,probs):
  def T2(x, prob, max):
    y = (2*x-prob[0])/(sum(prob[:max-1])+sum(prob[1:]))
    return y
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  df_out = df.copy()
  for j in range(n):
    prob = probs[j]
    max = maxs[j]
    if maxs[j] == 2:
      for i in range(m):
        if df.iloc[i,j] < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        else:
          df_out.iloc[i,j] = 2
    else:
      for i in range(m):
        r = df.iloc[i,j]
        if r < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        elif r > T2(sum(prob[:max-1]),prob,max):
          df_out.iloc[i,j] = max
        for k in range(1,max-1):
          if T2(sum(prob[:k]),prob,max) <= r < T2(sum(prob[:k+1]),prob,max):
            df_out.iloc[i,j] = k+1
  return df_out

In [130]:
# A subroutine that calculates $\tau_b$ coefficient for columns $i$ and $j$ of the input dataframe `df`.

def tau_b(df, i, j):
  import scipy.stats as stats
  dat = df.copy()
  keep_ind = []
  for k in range(len(df)):
    if df.isna().iloc[k,i]+df.isna().iloc[k,j] == 0:
      keep_ind.append(k)
  dat_out = dat.iloc[keep_ind,:]
  tau, p = stats.kendalltau(dat_out.iloc[:,i],dat_out.iloc[:,j])
  return tau

In [131]:
def tau_b_mat(df):
  import numpy as np
  n = df.shape[1]
  tau_mat = np.empty(shape=(n, n), dtype='double')
  for i in range(n):
    for j in range(n):
      tau_mat[i,j] = tau_b(df, i, j)
  return tau_mat

# Subroutine: Column-wise aggregation 
It takes a normalized dataframe `df` and the column label `i` and returns the aggregated column by weighted average:
$$
s_i = \frac{\sum_l w_l X_i}{\sum_l w_l}
$$`

In [132]:
def c_agg(df, w):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]

  agg_mat = np.empty(shape=(m, n), dtype='double')
  masked_df = np.ma.masked_array(df, np.isnan(df))

  for i in range(m):
    for j in range(n):
      agg_mat[i,j] = np.ma.average(masked_df[i,:], weights=w[:,j])
  return agg_mat

Subroutine: Take a column of a missing data and return the imputed one based on monotonization

In [133]:
def monotonization_old(input):

  import gurobipy as gp
  m = gp.Env(empty=True)
  m.setParam('WLSACCESSID', '54ea4edc-45f3-4ac2-b06f-cc4809a3c06c')
  m.setParam('WLSSECRET', '2802dfc5-13c3-4ceb-b6c2-b217b0d5994a')
  m.setParam('LICENSEID', 947226)
  m.setParam("OutputFlag", 0) #add by CHP
  m.start()

# Create the model within the Gurobi environment
  m = gp.Model(env=m)

  from gurobipy import GRB
  import numpy as np

  n = len(input)
  c = np.nanmax(input)
  k = n - np.isnan(input).sum()

  #print(input)
  ind = np.where(1-np.isnan(input))

  K = ind[0]


  #m = gp.Model('Monotonization')
  x = m.addVars(n, lb=1, ub=c, name="x")
  u = m.addVars(n, lb=0, name="u")
  w = m.addVars(n, lb=0, name="w")

  m.setObjective(gp.quicksum(u[i]+w[i] for i in K), GRB.MINIMIZE)

  m.addConstr(1 <= x[0])
  m.addConstr(x[n-1] <= c)
  m.addConstrs(x[i]<=x[i+1] for i in range(n-1))
  m.addConstrs(x[j]-input[j] == u[j]-w[j] for j in K)

  m.optimize()

  x_out = m.getAttr('X', x)

  for i in K:
    #print(i)
    x_out[i] = input[i]

  return x_out

In [134]:
# new optimization procedure (2023.04.18)
def monotonization(input):

  import numpy as np
  n = len(input)
  c = np.nanmax(input)
  # k = n - np.isnan(input).sum()
  input_obs = input[~np.isnan(input)]
  obs = len(input_obs)
  
  ind = np.where(1-np.isnan(input))
  K = ind[0]

  import gurobipy as gp
  m = gp.Env(empty=True)
  m.setParam('WLSACCESSID', '54ea4edc-45f3-4ac2-b06f-cc4809a3c06c')
  m.setParam('WLSSECRET', '2802dfc5-13c3-4ceb-b6c2-b217b0d5994a')
  m.setParam('LICENSEID', 947226)
  m.setParam("OutputFlag", 0) #add by CHP
  m.start()

# Create the model within the Gurobi environment
  m = gp.Model(env=m)

  from gurobipy import GRB

  y = m.addVars(obs, lb=1, ub=c, name="y")
  u = m.addVars(obs, lb=0, name="u")
  w = m.addVars(obs, lb=0, name="w")

  m.setObjective(gp.quicksum(u[i]+w[i] for i in range(obs)), GRB.MINIMIZE)

  m.addConstr(1 <= y[0])
  m.addConstr(y[obs-1] <= c)
  m.addConstrs(y[i] <= y[i+1] for i in range(obs-1))
  m.addConstrs(y[j]-input_obs[j] == u[j]-w[j] for j in range(obs))

  m.optimize()

  y_out = m.getAttr('X', y)

  x_out = input

  # print(x_out)
  # print(y_out)
  # print(K)

  for j in range(n):
      if j <= (K[0]+K[1])/2:
        x_out[j] = y_out[0]
      elif j > (K[obs-2]+K[obs-1])/2:
        x_out[j] = y_out[obs-1]
      else:
        for i in range(1,obs-1):
          if (K[i-1]+K[i])/2 < j <= (K[i]+K[i+1])/2:
            x_out[j] = y_out[i]
  
  for i in range(obs):
    x_out[K[i]] = input_obs[i]

  return x_out


In [135]:
# new modified optimization procedure (2023.04.25)
def monotonizationobs(input):

  import numpy as np
  n = len(input)
  c = np.nanmax(input)
  # k = n - np.isnan(input).sum()
  input_obs = input[~np.isnan(input)]
  obs = len(input_obs)
  
  ind = np.where(1-np.isnan(input))
  K = ind[0]

  import gurobipy as gp
  m = gp.Env(empty=True)
  # m.setParam('WLSACCESSID', '54ea4edc-45f3-4ac2-b06f-cc4809a3c06c')
  # m.setParam('WLSSECRET', '2802dfc5-13c3-4ceb-b6c2-b217b0d5994a')
  # m.setParam('LICENSEID', 947226)

  m.setParam('WLSACCESSID', 'd2d17a05-2f74-417a-8979-9a301e0be8b1')
  m.setParam('WLSSECRET', '73d51693-5359-4d7b-a92d-dba00689571f')
  m.setParam('LICENSEID', 2367307)
  m.setParam("OutputFlag", 0) #add by CHP
  m.start()

# Create the model within the Gurobi environment
  m = gp.Model(env=m)

  from gurobipy import GRB

  y = m.addVars(obs, lb=1, ub=c, name="y")
  u = m.addVars(obs, lb=0, name="u")
  w = m.addVars(obs, lb=0, name="w")

  m.setObjective(gp.quicksum(u[i]+w[i] for i in range(obs)), GRB.MINIMIZE)

  m.addConstr(1 <= y[0])
  m.addConstr(y[obs-1] <= c)
  m.addConstrs(y[i] <= y[i+1] for i in range(obs-1))
  m.addConstrs(y[j]-input_obs[j] == u[j]-w[j] for j in range(obs))

  m.optimize()

  y_out = m.getAttr('X', y)

  x_out = input

  for j in range(n):
    if j <= K[0]:
      x_out[j] = y_out[0]
    for k in range(obs-1):
      if j >= K[k] and j < K[k+1]:
        x_out[j] = y_out[k+1]
    if j >= K[k+1]:
      x_out[j] = y_out[obs-1]

  for i in range(obs):
    x_out[K[i]] = input_obs[i]

  return x_out


In [136]:
import numpy as np
inp = np.array([np.nan, 1, 1, np.nan, 2, 2, np.nan, 2, np.nan, 3, 3, np.nan, 3, np.nan, 4, 4, np.nan])
input = np.copy(inp)
output_old = monotonization_old(input)
input2 = np.copy(inp)
output = monotonization(input2)
print(np.array([input]))
# print(np.array([output_old]))
print(list(output_old.values()))
print(np.array([output]))

[[nan  1.  1. nan  2.  2. nan  2. nan  3.  3. nan  3. nan  4.  4. nan]]
[1.0, 1.0, 2.0, 2.0, 2.0, 3.0, 3.0, 3.0, 4.0, 4.0, 1.0, 1.0, 2.0, 3.0, 3.0, 4.0, 4.0]
[[1. 1. 1. 1. 2. 2. 2. 2. 2. 3. 3. 3. 3. 3. 4. 4. 4.]]


In [137]:
def rmse_cal(df, corr_mat, power):
  avail_ind = np.where(1-df.isna())
  avail_r_ind = avail_ind[0]
  avail_c_ind = avail_ind[1]
  l = len(avail_r_ind)
  masked_df = np.ma.masked_array(df, np.isnan(df))
  df_out = df.copy()
  for k in range(l):
    i = avail_r_ind[k]
    j = avail_c_ind[k]
    df_out.iloc[i,j] = np.ma.average(masked_df[i,:], weights=np.power(corr_mat[:,j],power))
  diff = df - df_out
  rmse = ((diff ** 2).sum().sum()/l) ** 0.5
  return rmse

In [138]:
def rmse_cal2(df, power):
  avail_ind = np.where(1-df.isna())
  avail_r_ind = avail_ind[0]
  avail_c_ind = avail_ind[1]
  l = len(avail_r_ind)

  masked_df = np.ma.masked_array(df, np.isnan(df))
  df_out = df.copy()

  w = np.array(df.corr())
  w[w<0]=0
  for i in range(len(w)):
    w[i,i]=0

  corr_mat = w

  for k in range(l):
    i = avail_r_ind[k]
    j = avail_c_ind[k]
    df_out.iloc[i,j] = np.ma.average(masked_df[i,:], weights=np.power(corr_mat[:,j],power))
  diff = df - df_out
  rmse = ((diff ** 2).sum().sum()/l) ** 0.5
  return rmse

In [139]:
def nested_quantile(df, a_min, a_max, n_level, RMSE_max):
  import numpy as np
  Q = np.zeros(5)
  Q[0] = a_min
  Q[4] = a_max
  beta = abs(Q[0]-Q[4])/4
  for j in range(1,4):
    Q[j] = Q[0] + j*beta

  # RMSE_L = rmse_calculator(df,Q[0])
  # RMSE_R = rmse_calculator(df,Q[4])
  RMSE_L = rmse_cal2(df,Q[0])
  RMSE_R = rmse_cal2(df,Q[4])

  # When RMSE_M=1, it is temporary
  RMSE_M = 1

  for i in range(n_level):

    RMSE = pd.DataFrame([[Q[0],RMSE_L],[Q[1],None],[Q[2],RMSE_M],[Q[3],None],[Q[4],RMSE_R]])

    for k in range(1,4):
      if k != 2 or RMSE_M == 1:
        # RMSE.iloc[k,1] =  rmse_calculator(df,Q[k])
        RMSE.iloc[k,1] =  rmse_cal2(df,Q[k])
      else:
        RMSE.iloc[k,1] = RMSE_M
    
    RMSE_sorted = RMSE.sort_values(by=[1])
        
    #print('RMSE Sorted:')
    #display(RMSE_sorted)
    
    alpha = RMSE_sorted.iloc[0,0]

    if RMSE_sorted.iloc[0,0] == Q[0]:
      Q[4] = Q[1]
      RMSE_R = RMSE.iloc[1,1] 
      RMSE_M = 1
    elif RMSE_sorted.iloc[0,0] == Q[4]:
      Q[0] = Q[3] 
      RMSE_L = RMSE.iloc[3,1]
      RMSE_M = 1
    elif RMSE_sorted.iloc[0,0] == Q[1]:
      Q[4] = Q[2]
      RMSE_M = RMSE.iloc[1,1]
      RMSE_R = RMSE.iloc[2,1]
    elif RMSE_sorted.iloc[0,0] == Q[2]:
      Q[0] = Q[1]
      Q[4] = Q[3]
      RMSE_L = RMSE.iloc[1,1]
      RMSE_M = RMSE.iloc[2,1]
      RMSE_R = RMSE.iloc[3,1]
    elif RMSE_sorted.iloc[0,0] == Q[3]:
      Q[0] = Q[2]
      RMSE_L = RMSE.iloc[2,1]
      RMSE_M = RMSE.iloc[3,1]

    beta = abs(Q[0]-Q[4])/4
    for j in range(1,4):
      Q[j] = Q[0] + j*beta

  return alpha

In [140]:
def power_finder(df, corr_mat, step_size, init_power, max_power):
  import math
  current_power = init_power
  opt_power = current_power
  opt_rmse = rmse_cal(df, corr_mat, opt_power)
  n_iter = math.floor((max_power-init_power)/step_size)
  for i in range(n_iter):
    current_power += step_size
    new_rmse = rmse_cal(df, corr_mat, current_power)
    if opt_rmse > new_rmse:
      opt_power = current_power
      opt_rmse = new_rmse
  return opt_power

In [141]:
df_orig = df_dict["orig"]
mm = df_orig.shape[0]
nn = df_orig.shape[1]

corr_mat = df_orig.corr()
import numpy as np
temp_corr = np.copy(corr_mat)

In [142]:
#@title Run MonoImoute algorithm
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

acc_list = []
rmse_list = []
mad_list = []
time_list = []

powers = [] #for monimpute

df_orig = df_dict["orig"]
df_orig.columns = range(df_orig.shape[1])

corr_mat = df_orig.corr()
temp_corr = np.copy(corr_mat)
temp_corr[temp_corr < 0] = 0
#print(np.round(temp_corr,2))
#print(np.round(corr_mat,2))
np.fill_diagonal(temp_corr, 0)
orig_corr = np.copy(temp_corr)


mm = df_orig.shape[0]
nn = df_orig.shape[1]

labs_test = labs[1:]

a_min = 0
a_max = 16
n_level = 4
RMSE_max = 1

count = 0
for lab in labs_test:

  print(lab)
  count += 1
  print(count)
  orig_corr = np.copy(temp_corr)
  df = df_dict[lab]
  import numpy as np
  masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
  #df, maxs = normal1(masked_df)
  #df, maxs = normal2(masked_df)
  #df, maxs, probs = normal3(masked_df)
  df_norm = normal3(masked_df)


  # from missingpy import MissForest
  # imputer = MissForest()

  # from fancyimpute import SoftImpute
  # imputer = SoftImpute()

  start = time_lib.time()
  #df_sampled = df_norm.sample(n=10)
  #df_norm = df
  #power = power_finder(df_sampled, orig_corr, 1, 1, 20)
  power = nested_quantile(df_norm, a_min, a_max, n_level, RMSE_max)
  powers.append(power)
  #power = 1

  #corr = np.power(orig_corr, power)  #issue: With irrational exponents 𝑟 and negative real numbers 𝑥 (2023.03.27)
  corr = np.power(orig_corr, power)  
  agg_mat = c_agg(df_norm, corr)

  df_agg_mat = pd.DataFrame(agg_mat)
  df_agg_mat.columns = ['agg'+str(i) for i in range(df.shape[1])]
  df_agg_mat.index = df.index
  df_agg = pd.concat([df, df_agg_mat], axis=1)
  #print(df_agg.shape)
  id_column = pd.DataFrame(range(mm))
  id_column.index = df_agg.index
  df_agg = pd.concat([id_column, df_agg], axis=1)
  #print(df_agg.shape)
  df_agg.columns = range(2*nn+1)
  
  

  for i in range(nn):
    df_agg_sorted = df_agg.sort_values(by=[df_agg.columns[nn+i+1],df_agg.columns[i+1]])
    index_i = df_agg_sorted.index
    vec = np.array(df_agg_sorted.iloc[:,i+1])
    out = monotonization(vec)
    out_list = []
    for i in range(len(out)):
      out_list.append(out[i])

    out_df = pd.DataFrame(out_list, index=index_i)
    df_agg = pd.concat([df_agg_sorted, out_df], axis=1)

  df_agg.columns = range(3*nn+1)
  df_agg_sorted_final = df_agg.sort_values(by=[df_agg.columns[0]])
  imputed = df_agg_sorted_final.iloc[:,2*nn+1:]
  
  #display(imputed)
  #save the result data
 
  # if count<10:
  #   filename = str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_mono_imputed.csv'
  # else:
  #   filename = str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_mono_imputed.csv'
    
  # imputed.to_csv(filename)
    
  imputed.columns = range(nn)

  end = time_lib.time()
  time = end - start

  df_orig.index = range(mm)
  imputed.index = range(mm)

  df_orig.columns = range(nn)
  imputed.columns = range(nn)

  diff = df_orig - imputed
  sq_diff = diff ** 2
  abs_diff = abs(diff)

  n_match = 0
  for i in range(mm):
    for j in range(nn):
      if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
        n_match += 1
  acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
  print(acc)  
  rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
  mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
  print(rmse)
  print(mad)
  print(time)
  acc_list.append(acc)
  rmse_list.append(rmse)
  mad_list.append(mad)
  time_list.append(time)

test01
1
0.5183987682832949
0.8682226808581898
0.5628945342571209
184.78397703170776
test02
2
0.517782909930716
0.855539392453691
0.5570438799076213
210.92979431152344
test03
3
0.5208622016936104
0.865647532486742
0.5602771362586605
184.66452980041504
test04
4
0.5241687192118226
0.8543931675341652
0.5514162561576355
186.16446375846863
test05
5
0.5170874384236454
0.8567322497625797
0.5591133004926109
185.84425354003906
test06
6
0.5375615763546798
0.8432395600688007
0.5364839901477833
188.13351726531982
test07
7
0.5221674876847291
0.8570017330457096
0.5537253694581281
183.7998869419098
test08
8
0.5210899014778325
0.8590649631066549
0.5541871921182266
151.28229546546936
test09
9
0.520935960591133
0.8573609123293106
0.5558805418719212
151.78315091133118
test10
10
0.5263238916256158
0.8509629433967631
0.5508004926108374
151.7848494052887


In [143]:
# print(sparsity)

print("Accuracy")
for i in acc_list:
  print(i)
print("\nRMSE")
for i in rmse_list:
  print(i)
print("\nMAD")
for i in mad_list:
  print(i)
print("\nTime")
for i in time_list:
  print(i)



Accuracy
0.5183987682832949
0.517782909930716
0.5208622016936104
0.5241687192118226
0.5170874384236454
0.5375615763546798
0.5221674876847291
0.5210899014778325
0.520935960591133
0.5263238916256158

RMSE
0.8682226808581898
0.855539392453691
0.865647532486742
0.8543931675341652
0.8567322497625797
0.8432395600688007
0.8570017330457096
0.8590649631066549
0.8573609123293106
0.8509629433967631

MAD
0.5628945342571209
0.5570438799076213
0.5602771362586605
0.5514162561576355
0.5591133004926109
0.5364839901477833
0.5537253694581281
0.5541871921182266
0.5558805418719212
0.5508004926108374

Time
184.78397703170776
210.92979431152344
184.66452980041504
186.16446375846863
185.84425354003906
188.13351726531982
183.7998869419098
151.28229546546936
151.78315091133118
151.7848494052887


In [144]:
df = pd.DataFrame(data=[acc_list, rmse_list,mad_list,time_list])
df.T

,0,1,2,3
0,0.518399,0.868223,0.562895,184.783977
1,0.517783,0.855539,0.557044,210.929794
2,0.520862,0.865648,0.560277,184.664530
3,0.524169,0.854393,0.551416,186.164464
4,0.517087,0.856732,0.559113,185.844254
5,0.537562,0.843240,0.536484,188.133517
6,0.522167,0.857002,0.553725,183.799887
7,0.521090,0.859065,0.554187,151.282295
8,0.520936,0.857361,0.555881,151.783151
9,0.526324,0.850963,0.550800,151.784849


In [145]:
print(powers)

[5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5]
